# Hooks Basics: Your First Hook and the Event Lifecycle

Hooks let you observe, extend, and control a Strands agent at runtime. Every agent invocation fires a sequence of typed events (`BeforeInvocationEvent`, `BeforeModelCallEvent`, `BeforeToolCallEvent`, and their `After*` counterparts), and each event gives a hook a chance to read agent state, modify specific writable fields, or short-circuit execution.

This notebook covers two foundational topics:

1. **Your first hook** - register a callback and measure model latency.
2. **The full event lifecycle** - print the exact order of every event across model and tool calls.

Every example is self-contained and runnable.

> **Next:** [02_hooks_intercept.ipynb](./02_hooks_intercept.ipynb) covers writable fields that let hooks block tools, retry calls, and auto-continue the agent.


## Lifecycle at a glance

```
agent("Summarize the weather in Seattle")
|
+-- BeforeInvocationEvent              <-- once per request
|   +-- MessageAddedEvent  (user message appended)
|
+-- BeforeModelCallEvent               <-- each model invocation
|   +-- AfterModelCallEvent            <-- writable: retry
|      +-- MessageAddedEvent (assistant turn)
|
+-- BeforeToolCallEvent                <-- each requested tool
|   |                                      writable: cancel_tool, selected_tool, tool_use
|   +-- AfterToolCallEvent             <-- writable: result, retry   (reverse order)
|      +-- MessageAddedEvent (tool result)
|
+-- (model -> tools loop until end_turn)
|
+-- AfterInvocationEvent               <-- once per request (reverse order)
                                           writable: resume
```

`After*` callbacks fire in **reverse** registration order (the same inversion pattern as `try/finally`), so an outer observer sees what an inner one changed.


## Setup

This tutorial uses Claude Haiku 4.5 on Amazon Bedrock by default. It works with any Strands-supported model. Swap the `model=` argument in the `Agent(...)` constructor if you prefer a different provider (see [Model Providers](https://strandsagents.com/docs/user-guide/concepts/model-providers/)).


In [ ]:
%pip install -r requirements.txt --quiet

In [ ]:
import logging
import time

from strands import Agent, tool
from strands.hooks import (
    AgentInitializedEvent,
    BeforeInvocationEvent,
    AfterInvocationEvent,
    BeforeModelCallEvent,
    AfterModelCallEvent,
    BeforeToolCallEvent,
    AfterToolCallEvent,
    MessageAddedEvent,
    HookProvider,
    HookRegistry,
)

# Use the same Bedrock inference profile as the other 01-learn tutorials.
MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

# Quiet SDK logs so the hook demonstrations stay the focus of the output.
logging.getLogger("strands").setLevel(logging.WARNING)

## 1. Your first hook: `BeforeModelCallEvent` and `AfterModelCallEvent`

The simplest hook is a plain function that takes an event argument. Register it with `agent.add_hook(callback)`. Strands infers the event type from your type hint.

Below we register a pair of hooks that time every model call.


In [ ]:
model_timings: list[float] = []

def start_timer(event: BeforeModelCallEvent) -> None:
    # Stash the start time on the event's invocation_state so the matching
    # After* hook can read it. invocation_state is a per-request dict.
    event.invocation_state["_model_start"] = time.perf_counter()

def record_duration(event: AfterModelCallEvent) -> None:
    start = event.invocation_state.get("_model_start")
    if start is not None:
        model_timings.append(time.perf_counter() - start)

agent = Agent(model=MODEL_ID, callback_handler=None)
agent.add_hook(start_timer)
agent.add_hook(record_duration)

result = agent("In one sentence, what is a hook?")
print("model calls:", len(model_timings))
print("avg latency:", f"{sum(model_timings) / len(model_timings):.2f}s")
print("answer:", str(result).strip())

Two things worth noticing:

* `agent.add_hook(callback)` uses the callback's type annotation (`event: BeforeModelCallEvent`) to bind the event type. You can also pass the type explicitly: `agent.add_hook(start_timer, BeforeModelCallEvent)`.
* We shared data between the two hooks through `event.invocation_state`, a dictionary that lives for the duration of one `agent(...)` call. We will return to this pattern in [03_hooks_packaging.ipynb](./03_hooks_packaging.ipynb).


Two other things worth knowing about registration:

* **`AgentInitializedEvent`** fires once at the end of the `Agent(...)` constructor. It is the right place for one-time setup that needs a fully constructed agent.
* `HookProvider` instances can be attached either via the constructor (`Agent(hooks=[MyProvider()])`) or **after construction** with `agent.hooks.add_hook(MyProvider())`. The next cell shows both.


In [ ]:
class StartupLogger(HookProvider):
    """Run once when the agent is fully constructed."""

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(AgentInitializedEvent, self._on_init)

    def _on_init(self, event: AgentInitializedEvent) -> None:
        print(f"agent initialized: model={type(event.agent.model).__name__}")

# Option A: pass at construction time.
agent_a = Agent(model=MODEL_ID, hooks=[StartupLogger()], callback_handler=None)

# Option B: attach after construction with agent.hooks.add_hook.
agent_b = Agent(model=MODEL_ID, callback_handler=None)
agent_b.hooks.add_hook(StartupLogger())  # Note: AgentInitializedEvent has already
                                          # fired for agent_b, so this callback will
                                          # not run. Construction-time registration
                                          # is required to observe initialization.

## 2. The full event lifecycle

To see the real ordering, attach a hook to every event type and print them in the order they fire. We will also add a tool so the tool events actually emit.


In [ ]:
@tool
def get_weather(city: str) -> str:
    """Return the current weather for a city.

    Args:
        city: City name.
    """
    fake = {"Seattle": "rainy, 52F", "Austin": "sunny, 78F"}
    return fake.get(city, "clear, 70F")

trace: list[str] = []

def log(name):
    def cb(event):
        trace.append(name)
    return cb

agent = Agent(model=MODEL_ID, tools=[get_weather], callback_handler=None)

# Register one logger per event type. add_hook also accepts (callback, EventType).
for event_type in (
    BeforeInvocationEvent,
    MessageAddedEvent,
    BeforeModelCallEvent,
    AfterModelCallEvent,
    BeforeToolCallEvent,
    AfterToolCallEvent,
    AfterInvocationEvent,
):
    agent.add_hook(log(event_type.__name__), event_type)

agent("What's the weather in Seattle?")

for i, name in enumerate(trace):
    print(f"{i:2d}. {name}")

You should see the ordering promised at the top of the notebook:

```
BeforeInvocationEvent
MessageAddedEvent            <-- user turn
BeforeModelCallEvent
AfterModelCallEvent
MessageAddedEvent            <-- assistant turn requesting the tool
BeforeToolCallEvent
AfterToolCallEvent
MessageAddedEvent            <-- tool result
BeforeModelCallEvent         <-- model sees the tool result, produces the final answer
AfterModelCallEvent
MessageAddedEvent
AfterInvocationEvent
```

The model/tool cycle repeats as many times as the model needs before stopping. `AfterInvocationEvent` fires exactly once at the end.


## Recap

You now know:

- How to register a hook four ways: `agent.add_hook(fn)` (type inferred), `agent.add_hook(fn, EventType)`, `hooks=[Provider()]` in the constructor, and `agent.hooks.add_hook(Provider())` after construction.
- The exact order of hook events in a Strands invocation, including the `After*` reverse-ordering rule.
- That `AgentInitializedEvent` fires once at construction time for one-time setup.

### Next

- [**02_hooks_intercept.ipynb**](./02_hooks_intercept.ipynb) - writable fields that let hooks block tools, retry failed calls, auto-continue the agent, and redact input.
- [**03_hooks_packaging.ipynb**](./03_hooks_packaging.ipynb) - cross-hook data sharing with `invocation_state` and bundling hooks into reusable `HookProvider` classes.
